# Music Store Customer Support Agent

A LangChain agent that answers questions about a customer's purchase history using the Chinook SQLite database.
The authenticated customer is passed in through the runtime context, so the agent can only query that customer's invoices.

In [1]:
from dotenv import load_dotenv
from dataclasses import dataclass

load_dotenv(override=True)

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.utilities import SQLDatabase
from langgraph.runtime import get_runtime

/var/folders/m6/yxk9f4b54zq2phytb2t66b700000gn/T/ipykernel_3517/1124902494.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Database

In [2]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
print(db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [3]:
@dataclass
class RuntimeContext:
    db: SQLDatabase
    customer_id: int = 1

In [4]:
@tool
def get_purchase_history() -> str:
    """Get the authenticated customer's invoice history."""

    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db
    customer_id = runtime.context.customer_id

    query = """
    SELECT
        InvoiceId,
        InvoiceDate,
        Total
    FROM Invoice
    WHERE CustomerId = :customer_id
    ORDER BY InvoiceDate DESC
    """

    return db.run(
        query,
        parameters={"customer_id": customer_id},
    )

In [5]:
agent = create_agent(
    model="anthropic:claude-sonnet-4-5",
    tools=[get_purchase_history],
    system_prompt=(
        "You are a customer support agent for a music store. "
        "Only help the authenticated customer with their own account."
    ),
    context_schema=RuntimeContext,
)

In [6]:
for step in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What purchases have I made?"
            }
        ]
    },
    context=RuntimeContext(
        db=db,
        customer_id=1,
    ),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What purchases have I made?
================================== Ai Message ==================================

[{'text': "I'll retrieve your purchase history for you.", 'type': 'text'}, {'id': 'toolu_01XmykwujE12rJsC868uALgf', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_purchase_history', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  get_purchase_history (toolu_01XmykwujE12rJsC868uALgf)
 Call ID: toolu_01XmykwujE12rJsC868uALgf
  Args:
================================= Tool Message =================================
Name: get_purchase_history

[(382, '2013-08-07 00:00:00', 8.91), (327, '2012-12-07 00:00:00', 13.86), (316, '2012-10-27 00:00:00', 1.98), (195, '2011-05-06 00:00:00', 0.99), (143, '2010-09-15 00:00:00', 5.94), (121, '2010-06-13 00:00:00', 3.96), (98, '2010-03-11 00:00:00', 3.98)]
================================== Ai Message ==================================

Here's your p